In [6]:
# get rid of black images
# maybe contrast
# maybe data augmentation
# maybe get rid of too blurry images

In [ ]:
from pathlib import Path

DATA_RAW = Path("datasets/raw")
DATA_PROCESSED = Path("datasets/processed")
IMAGES_DIR = "images"
LABELS_DIR = "labels"
SPLITS = ["train", "val", "test"]


In [ ]:
#delete black and blurry images
import shutil
import cv2
import numpy as np

BLACK_PIXEL_THRESHOLD = 0.93  # 93% pixels near black
BLUR_THRESHOLD = 40


In [16]:
def is_mostly_black(image_path: Path, threshold=BLACK_PIXEL_THRESHOLD) -> bool:
    img = cv2.imread(str(image_path))
    if img is None:
        return True  # treat unreadable images as invalid
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # count near-black pixels
    black_pixels = np.sum(gray < 10)  # intensity < 10 = almost black
    total_pixels = gray.size

    ratio = black_pixels / total_pixels
    return ratio >= threshold


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

In [17]:
import cv2

def is_blurry(image_path, threshold=BLUR_THRESHOLD):
    img = cv2.imread(str(image_path))
    
    if img is None:
        return True

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Laplacian variance (sharpness measure)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()

    return variance < threshold

In [18]:
def is_bad_image(image_path):
    return is_mostly_black(image_path) or is_blurry(image_path)

In [19]:
blurr_amount = 0
black_amount = 0
for split in SPLITS:
    img_src_dir = DATA_RAW / IMAGES_DIR / split
    lbl_src_dir = DATA_RAW / LABELS_DIR / split

    img_dst_dir = DATA_PROCESSED / IMAGES_DIR / split
    lbl_dst_dir = DATA_PROCESSED / LABELS_DIR / split

    ensure_dir(img_dst_dir)
    ensure_dir(lbl_dst_dir)

    for img_path in img_src_dir.glob("*.jpg"):
        name = img_path.stem
        lbl_path = lbl_src_dir / f"{name}.txt"

        if is_mostly_black(img_path):
            print(f"Skipping (black): {img_path}")
            black_amount += 1
            continue
        
        if is_blurry(img_path):
            print(f"Skipping (blurry): {img_path}")
            blurr_amount += 1
            continue
        # if is_bad_image(img_path):
        #     print(f"Skipping (bad): {img_path}")
        #     continue

        shutil.copy2(img_path, img_dst_dir / img_path.name)

        if lbl_path.exists():
            shutil.copy2(lbl_path, lbl_dst_dir / lbl_path.name)
        else:
            print(f"Warning: missing label for {img_path.name}")

print(f"Total images skipped (black): {black_amount}")
print(f"Total images skipped (blurry): {blurr_amount}")

Skipping (bad): datasets\raw\images\val\10_2324.jpg
Skipping (bad): datasets\raw\images\val\10_2338.jpg
Skipping (bad): datasets\raw\images\val\10_2339.jpg
Skipping (bad): datasets\raw\images\val\10_2344.jpg
Skipping (bad): datasets\raw\images\val\10_2354.jpg
Skipping (bad): datasets\raw\images\val\10_2364.jpg
Skipping (bad): datasets\raw\images\val\10_2374.jpg
Skipping (bad): datasets\raw\images\val\10_2384.jpg
Skipping (bad): datasets\raw\images\val\10_2394.jpg
Skipping (bad): datasets\raw\images\val\10_2398.jpg
Skipping (bad): datasets\raw\images\val\10_2404.jpg
Skipping (bad): datasets\raw\images\val\10_2406.jpg
Skipping (bad): datasets\raw\images\val\10_2408.jpg
Skipping (bad): datasets\raw\images\val\10_2414.jpg
Skipping (bad): datasets\raw\images\val\10_2434.jpg
Skipping (bad): datasets\raw\images\val\10_2444.jpg
Skipping (bad): datasets\raw\images\val\10_2454.jpg
Skipping (bad): datasets\raw\images\val\10_2464.jpg
Skipping (bad): datasets\raw\images\val\10_2474.jpg
Skipping (ba

In [20]:
print("PreProcessing completed - stats:")

def count_images(base_path):
    counts = {}
    for split in SPLITS:
        path = base_path / IMAGES_DIR / split
        counts[split] = len(list(path.glob("*.jpg")))
    return counts


raw_counts = count_images(DATA_RAW)
processed_counts = count_images(DATA_PROCESSED)



for split in SPLITS:
    before = raw_counts.get(split, 0)
    after = processed_counts.get(split, 0)
    print(f"{split.upper():5s} | before: {before:5d} | after: {after:5d}")

PreProcessing completed - stats:
VAL   | before:  2384 | after:   531
